In [ ]:
import pandas as pd
import altair as alt

In [ ]:
sge_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251202_SGEsubset.xlsx')

In [ ]:
gene = 'BARD1'
df = sge_ppj_subset.loc[sge_ppj_subset['Gene'] == gene]
df.head()
alt.data_transformers.disable_max_rows() #gets rid of max data length problem

In [ ]:
df = df.dropna(subset = ['simplified_consequence'])
df = df.rename(columns = {'simplified_consequence': 'Consequence',
                     'auth_reported_score': 'score',
                      'auth_reported_func_class': 'functional_consequence',
                        'clinvar_sig': 'Germline classification',
                     'ref_allele': 'ref'})
df.loc[df['Consequence'].str.contains('missense'), 'Consequence'] = 'Missense'
df.loc[df['Consequence'] == 'synonymous_variant', 'Consequence'] = 'Synonymous'
df.loc[df['Consequence'] == 'intron_variant', 'Consequence'] = 'Intron'
df.loc[df['Consequence'] == 'stop_gained', 'Consequence'] = 'Stop Gained'
df.loc[df['Consequence'] == 'stop_lost', 'Consequence'] = 'Stop Lost'
df.loc[df['Consequence'].str.contains('site'), 'Consequence'] = 'Canonical Splice'
df.loc[df['Consequence'].str.contains('ing_var'), 'Consequence'] = 'Splice Region'
df.loc[df['Consequence'].str.contains('UTR'), 'Consequence'] = 'UTR Variant'
df.loc[df['Consequence'] == 'start_lost', 'Consequence'] = 'Start Lost'
df.loc[df['ref'].str.len() > 1, 'Consequence'] = '3bp Deletion'

df.head()

In [ ]:
def get_gmm_threshold(df):
    
    # find the GMM thresholds
    target_value = 0.950

    ab_df = df.loc[df['functional_consequence'].isin(['functionally_abnormal'])]
    norm_df = df.loc[df['functional_consequence'].isin(['functionally_normal'])]

    
    
    '''
    # Calculate the absolute difference for the Normal (N) density
    diffN = (norm_df['gmm_density_normal'] - target_value).abs()
    # Find the index of the minimum difference
    closest_index = diffN.idxmin()
    # Retrieve the row with the closest value
    closest_row_n = norm_df.loc[closest_index]
    
    # now repeat that for the abnormal density
    # Calculate the absolute difference
    diffA = (ab_df['gmm_density_abnormal'] - target_value).abs()
    # Find the index of the minimum difference
    closest_index = diffA.idxmin()
    # Retrieve the row with the closest value
    closest_row_a = ab_df.loc[closest_index]
    '''
    # now we get the scores that are the closest to the (n)ormal and (a)bnormal thresholds
    upprthresh = norm_df['score'].min()
    lwrthresh = ab_df['score'].max()


    thresholds = [lwrthresh, upprthresh]

    return thresholds

In [ ]:
print(df.columns)
thresholds = get_gmm_threshold(df)

std_height = 150
std_width = 500
plot_domain = [-0.45, 0.1]
tick_values = [-0.4, -0.3, -0.2, -0.1, 0, 0.1]
title_fontsize = 16
length = len(df)

palette = [
    '#006616', # dark green,
    '#81B4C7', # dusty blue
    '#ffcd3a', # yellow
    '#6AA84F', # med green
    '#93C47D', # light green
    '#888888', # med gray
    '#000000', # black
    '#1170AA', # darker blue
    '#CFCFCF' # light gray  
]


variant_types = [
    'Synonymous',
    'Missense',  
    'Stop Gained',
    'Intron', 
    'UTR Variant',
    'Stop Lost',
    'Start Lost',
    'Canonical Splice', 
    'Splice Region'
]

chart = alt.Chart(df).mark_bar().encode(
    alt.X('score', 
          axis = alt.Axis(title = '', 
                          labels = False,
                          values= tick_values
                         ), 
          scale = alt.Scale(domain = plot_domain),
          bin = alt.Bin(maxbins = 50)),
    alt.Y('count()', 
          axis = alt.Axis(title = '# of Vars.', 
                          labelFontSize = 16, 
                          titleFontSize = 20)),
    color = alt.Color('Consequence:N', 
                 scale = alt.Scale(range = palette,
                                  domain = variant_types), 
                 legend = alt.Legend(titleFontSize = 14, 
                                     labelFontSize = 12,
                                     orient = 'top-left',
                                     columns = 2,
                                    offset = 10))
).properties(
    height = std_height,
    width = std_width,
    title = alt.TitleParams(text = f' {gene} (n = {length})',
                            fontSize = title_fontsize,
                            dy = 30
                           )
)

nf_line = alt.Chart(pd.DataFrame({'Functional Score': [thresholds[0]]})).mark_rule(color = 'red').encode(
x = 'Functional Score')

func_line = alt.Chart(pd.DataFrame({'Functional Score': [thresholds[1]]})).mark_rule(color = 'blue').encode(
x = 'Functional Score')


base_histo = alt.layer(chart, nf_line, func_line)

base_histo.display()

In [ ]:
w_clinvar_df = (df.dropna(subset = ['Germline classification'])).copy()

w_clinvar_df = w_clinvar_df.loc[w_clinvar_df['functional_consequence'].isin(['functionally_normal', 'functionally_abnormal'])]

w_clinvar_df = w_clinvar_df.loc[~w_clinvar_df['Germline classification'].isin(['Uncertain significance', 'Conflicting classifications of pathogenicity'])]

w_clinvar_df.loc[(w_clinvar_df['Germline classification'] == 'Benign') | (w_clinvar_df['Germline classification'] == 'Likely benign') | (w_clinvar_df['Germline classification'] == 'Benign/Likely benign'), 'simple_classification'] = 'Benign'
w_clinvar_df.loc[(w_clinvar_df['Germline classification'] == 'Pathogenic') | (w_clinvar_df['Germline classification'] == 'Likely pathogenic') | (w_clinvar_df['Germline classification'] == 'Pathogenic/Likely pathogenic'), 'simple_classification'] = 'Pathogenic'

w_clinvar_df = w_clinvar_df.loc[w_clinvar_df['simple_classification'].isin(['Benign',  'Pathogenic'])]

clinvar_histo = alt.Chart(w_clinvar_df).mark_bar().encode(
    x = alt.X('score',
     bin = alt.Bin(maxbins = 50)),
    y = 'count()',
    color = 'simple_classification'
)

clinvar_histo.display()

In [ ]:
density_plot = alt.Chart(w_clinvar_df).transform_density(
    'score',
    as_=['score', 'density'],
    groupby=['simple_classification'],
    extent=plot_domain
).mark_line(strokeWidth=2).encode(
    x=alt.X('score:Q', scale=alt.Scale(domain=plot_domain)),
    y=alt.Y('density:Q', axis = None),
    color=alt.Color('simple_classification:N',
                   scale = alt.Scale(
                       domain = ['Pathogenic', 'Benign'],
                       range = ['#CA7682','#1D7AAB']
                   ),
                  legend = None
                   )
)

density_plot.display()
# Layer with independent y-axes
chart = alt.layer(
    base_histo,density_plot, nf_line, func_line, 
).resolve_scale(
    y='independent',
    color = 'independent'
).configure_axis(
    grid = False
).configure_view(
    stroke = None
)

chart.display()